# HFusionHub RAG 评估 Notebook

此 Notebook 演示如何对 HFusionHub 的 RAG 检索引擎进行离线评估。

**功能**:
- 加载评估数据集（JSONL 格式）
- 调用 Python AI 服务的评估 API
- 计算 Recall@K、MRR、Hit Rate 等指标
- 可视化评估结果
- 分析失败案例

**前置条件**:
- Python AI 服务运行在 localhost:9000
- 至少有一个已索引的知识库

In [ ]:
import json
import requests
import time
from pathlib import Path
from typing import List, Dict, Any
from dataclasses import dataclass, field

# 配置
PYTHON_AI_URL = "http://localhost:9000"
INTERNAL_TOKEN = "dev-internal-token-change-me"
DATASET_PATH = Path("../python-ai/scripts/eval_example.jsonl")
KNOWLEDGE_BASE_ID = 1  # 替换为你的知识库 ID

HEADERS = {"X-Internal-Token": INTERNAL_TOKEN, "Content-Type": "application/json"}

print(f"Python AI URL: {PYTHON_AI_URL}")
print(f"Dataset: {DATASET_PATH}")

In [ ]:
@dataclass
class EvalResult:
    """单条评估结果"""
    query: str
    recall_at_5: float
    recall_at_10: float
    mrr: float
    hit_at_1: bool
    hit_at_5: bool
    latency_ms: float
    retrieved_docs: List[str] = field(default_factory=list)
    error: str = None


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    """加载 JSONL 格式的评估数据集"""
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


def evaluate_single(sample: Dict, kb_id: int, top_k: int = 10) -> EvalResult:
    """对单条样本执行检索评估"""
    query = sample["query"]
    expected_docs = set(sample.get("expected_documents", []))
    
    start = time.time()
    try:
        resp = requests.post(
            f"{PYTHON_AI_URL}/api/rag/debug-search",
            json={
                "query": query,
                "knowledge_base_id": kb_id,
                "top_k": top_k,
            },
            headers=HEADERS,
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", data.get("chunks", []))
    except Exception as e:
        return EvalResult(query=query, recall_at_5=0, recall_at_10=0,
                         mrr=0, hit_at_1=False, hit_at_5=False,
                         latency_ms=0, error=str(e))
    
    latency_ms = (time.time() - start) * 1000
    
    # 提取检索到的文档名
    retrieved_docs = []
    for r in results:
        meta = r.get("metadata", {})
        doc_name = meta.get("document_title", "") or r.get("document_name", "")
        retrieved_docs.append(doc_name)
    
    # 如果没有预期文档，只计算检索是否非空
    if not expected_docs:
        has_results = len(results) > 0
        return EvalResult(
            query=query, recall_at_5=float(has_results),
            recall_at_10=float(has_results), mrr=1.0 if has_results else 0.0,
            hit_at_1=has_results, hit_at_5=has_results,
            latency_ms=latency_ms, retrieved_docs=retrieved_docs
        )
    
    # 计算指标
    top5 = results[:5]
    top10 = results[:10]
    
    # Recall@K: 预期文档中至少有一个在 top-K 中
    top5_docs = {r.get("document_name", "") for r in top5}
    top10_docs = {r.get("document_name", "") for r in top10}
    recall_at_5 = 1.0 if expected_docs & top5_docs else 0.0
    recall_at_10 = 1.0 if expected_docs & top10_docs else 0.0
    
    # MRR: 第一个相关文档的排名倒数
    mrr = 0.0
    for i, r in enumerate(results, 1):
        doc_name = r.get("document_name", "")
        if doc_name in expected_docs:
            mrr = 1.0 / i
            break
    
    # Hit@K
    hit_at_1 = any(r.get("document_name", "") in expected_docs for r in results[:1])
    hit_at_5 = any(r.get("document_name", "") in expected_docs for r in results[:5])
    
    return EvalResult(
        query=query, recall_at_5=recall_at_5, recall_at_10=recall_at_10,
        mrr=mrr, hit_at_1=hit_at_1, hit_at_5=hit_at_5,
        latency_ms=latency_ms, retrieved_docs=retrieved_docs
    )

In [ ]:
def run_evaluation(samples: List[Dict], kb_id: int, top_k: int = 10) -> List[EvalResult]:
    """运行完整评估"""
    results = []
    total = len(samples)
    print(f"开始评估 {total} 条样本 (KB ID: {kb_id})...")
    
    for i, sample in enumerate(samples):
        result = evaluate_single(sample, kb_id, top_k)
        results.append(result)
        status = "❌" if result.error else ("✅" if result.hit_at_5 else "⚠️")
        print(f"  [{i+1}/{total}] {status} {result.query[:50]}... (MRR:{result.mrr:.2f}, {result.latency_ms:.0f}ms)")
    
    return results


def summarize(results: List[EvalResult]) -> Dict[str, float]:
    """汇总评估指标"""
    valid = [r for r in results if not r.error]
    n = len(valid)
    if n == 0:
        return {}
    
    return {
        "样本数": n,
        "Recall@5": sum(r.recall_at_5 for r in valid) / n,
        "Recall@10": sum(r.recall_at_10 for r in valid) / n,
        "MRR": sum(r.mrr for r in valid) / n,
        "Hit@1": sum(1 for r in valid if r.hit_at_1) / n,
        "Hit@5": sum(1 for r in valid if r.hit_at_5) / n,
        "平均延迟(ms)": sum(r.latency_ms for r in valid) / n,
        "错误数": len(results) - n,
    }


def show_failures(results: List[EvalResult], top_n: int = 5):
    """展示失败案例"""
    failures = [r for r in results if not r.error and r.mrr == 0.0]
    print(f"\n### 失败案例 (MRR=0): {len(failures)}/{len(results)}")
    for r in failures[:top_n]:
        print(f"  - 查询: {r.query[:80]}")
        if r.retrieved_docs:
            print(f"    检索到: {r.retrieved_docs[:3]}")

In [ ]:
# 加载数据集
samples = load_dataset(DATASET_PATH)
print(f"加载了 {len(samples)} 条评估样本")
for s in samples[:3]:
    print(f"  - {s['query'][:60]}...")

# 运行评估
results = run_evaluation(samples, KNOWLEDGE_BASE_ID)

# 汇总指标
metrics = summarize(results)
print("\n### 评估指标汇总")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# 失败案例
show_failures(results)

# 目标参考值:
print("\n### 目标参考值")
print("  Recall@5  >= 0.90")
print("  MRR       >= 0.70")
print("  Hit@5     >= 0.90")
print("  延迟      <  500ms")

## 评估结果分析

| 指标 | 参考目标 | 说明 |
|------|---------|------|
| Recall@5 | ≥ 0.90 | Top-5 中至少命中一个相关文档的比例 |
| MRR | ≥ 0.70 | 第一个相关文档的平均排名倒数 |
| Hit@5 | ≥ 0.90 | Top-5 中文档名匹配的比例 |
| 延迟 | < 500ms | 单次检索的平均响应时间 |

**常见优化方向**:
- Recall 低 → 开启 `RAG_HYBRID_ENABLED=true` 混合检索
- MRR 低 → 开启 `RAG_RERANKER_MODE=lexical` 重排序
- 延迟高 → 检查 embedding 服务可用性，减小候选集 `RAG_RETRIEVAL_MAX_CANDIDATES`